# Run on Kaggle/Colab

In [1]:
# !wget https://s3-eu-west-1.amazonaws.com/pfigshare-u-files/12785291/traintest_hcd.hdf5

In [2]:
# !ls /kaggle/input/datasets/bangth/intensity-gen-fm-lib

In [3]:
# !pip install -r /kaggle/input/datasets/bangth/intensity-gen-fm-lib/requirements.txt -q

In [4]:
# !pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu124 -q

## Lib integration

In [5]:
import sys
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching")
sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching\demo-code\2d")
import h5py
from collections import defaultdict, Counter
import numpy as np
from rich import print

In [6]:
file_path = r"E:\Dai hoc\2526I\dacn\flow-matching\data\traintest_hcd.hdf5"

## Open data

In [7]:
with h5py.File(file_path, "r") as f:
    print("Keys:", list(f.keys()))

    seqs = f["sequence_integer"][:]
    charges_oh = f["precursor_charge_onehot"][:]
    intensities = f["intensities_raw"][:]
    masses_raw = f["masses_raw"][:1]
    masses_pred = f["masses_pred"][:1]
    rev = f["reverse"][:1]
    rawfile = f["rawfile"][:1]
    collision_energy = f["collision_energy_aligned_normed"][:10]
    seq_ohs = f["sequence_onehot"][:25]

Keys:
[
    'collision_energy',
    'collision_energy_aligned',
    'collision_energy_aligned_normed',
    'intensities_raw',
    'masses_pred',
    'masses_raw',
    'method',
    'precursor_charge_onehot',
    'rawfile',
    'reverse',
    'scan_number',
    'score',
    'sequence_integer',
    'sequence_onehot'
]

In [8]:
charges_oh[0]

array([0, 0, 1, 0, 0, 0])

In [10]:
#  get the max vocab in seqs
max_vocab = int(np.max(seqs))
print("Max vocab in seqs:", max_vocab)

Max vocab in seqs: 21

In [ ]:
masses_raw[0]

array([ 1.75118622e+02,  0.00000000e+00,  0.00000000e+00,  1.87087021e+02,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  3.16128723e+02,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  4.15197174e+02,
        0.00000000e+00,  0.00000000e+00,  4.04227631e+02,  0.00000000e+00,
        0.00000000e+00,  5.78259155e+02,  0.00000000e+00,  0.00000000e+00,
        4.91256134e+02,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  5.90329041e+02,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        7.03412659e+02,  0.00000000e+00,  2.35144150e+02,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        2.68163849e+02,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  

In [ ]:
len(seqs)

In [ ]:
charges = np.argmax(charges_oh, axis=1) + 1
del charges_oh

In [ ]:
seqs[0]

### Utils

### Peak data

In [ ]:
# np.argmax(charges_oh, axis=1) + 1

In [ ]:
intensities

In [ ]:
# max_index = 0
# for seq in seqs:
#     for token in seq:
#         if token > max_index:
#             max_index = token
# print("Max token index:", max_index)

## Explore data

In [ ]:
max(1,2)

In [ ]:
min_charge = 10
max_charge = 0

for charge in charges:
    min_charge = min(min_charge, charge)
    max_charge = max(charge, max_charge)

print(f"Min charge: {min_charge}")
print(f"Max charge: {max_charge}")

## FLow matching training

In [ ]:
import torch
torch.set_default_device("cuda")
from torch import nn
# from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import imageio
import math

from gen_path import get_xt
from models import HCDFlowResMLP, HCDFlow

### Training configuration

In [ ]:
epoch = 2
batch_size = 256
model_path = r"E:\Dai hoc\2526I\dacn\flow-matching\run_real_data\checkpoints\tfmemb_adalm8_512_8e.pth"
model = HCDFlowResMLP(noise_dim=174, pep_dim=256, time_dim=128, charge_dim=9, num_blocks=8, min_charge=1, max_charge=6)
optimizer = torch.optim.AdamW(model.parameters(), eps=1e-8, lr=2e-4,weight_decay=1e-3)
model.load_state_dict(torch.load(model_path))

<All keys matched successfully>

#### Tranable params

In [ ]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# load pretrain
# model.load_state_dict(torch.load(r"E:\Dai hoc\2526I\dacn\flow-matching\run_real_data\models\pep=256\blocks=12\model_6e_1024b.pth"))

In [ ]:
loss_history = []
last_100_loss = []

In [ ]:
torch.tensor([[1, 2]]).resize(*torch.tensor([1,2]).shape).shape

In [ ]:
from tqdm.auto import tqdm
# pbar = tqdm(range(int(epoch)), desc="Training")

pbar = tqdm(range(int(epoch)), desc="Training")
num_samples = len(seqs)
num_batches = math.ceil(num_samples / batch_size)
for ep in pbar:
    model.train()

    for b in range(num_batches):
        start = b * batch_size
        end = min((b + 1) * batch_size, num_samples)

        batch_intensities = torch.tensor(
            intensities[start:end], dtype=torch.float32
        )
        batch_pep_seq = torch.tensor(
            seqs[start:end], dtype=torch.long
        )
        batch_charge = torch.tensor(
            charges[start:end], dtype=torch.long
        ).unsqueeze(1)

        cur_bs = batch_intensities.shape[0]

        noise = torch.randn_like(batch_intensities)
        t = torch.rand(cur_bs, 1)

        x_t = get_xt(batch_intensities, noise, t)
        u_pred = model(x_t, t=t, pep_seq=batch_pep_seq, charge=batch_charge)

        loss = nn.MSELoss()(u_pred, batch_intensities - noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        last_100_loss.append(loss.item())

        if len(last_100_loss) == 100:
            mean_last_100 = sum(last_100_loss) / 100
            last_100_loss.clear()
            loss_history.append(mean_last_100)

            pbar.set_postfix({
                "Last100": f"{mean_last_100:.4f}",
                "Avg": f"{(sum(loss_history)/len(loss_history)):.4f}",
            })
            if len(loss_history) % 100 == 0:
                print(f"Avg loss from last 10000 batch: {(sum(loss_history[-100:-1])/100):.4f}")


In [ ]:
open("loss_history.txt", "w+").write("\n".join(loss_history))

In [ ]:
def plot_loss_history(loss_history, smooth_window=None):
    """
    Plot training loss history.

    Args:
        loss_history (list or array): Danh sách loss theo từng step/epoch.
        smooth_window (int, optional): Nếu truyền vào, sẽ vẽ thêm đường smooth
                                       bằng moving average với window này.
    """
    plt.figure()
    plt.plot(loss_history)

    if smooth_window is not None and smooth_window > 1:
        import numpy as np
        loss_array = np.array(loss_history)
        kernel = np.ones(smooth_window) / smooth_window
        smooth_loss = np.convolve(loss_array, kernel, mode='valid')
        plt.plot(range(smooth_window - 1, len(loss_history)), smooth_loss)

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss History")
    plt.show()

plot_loss_history(loss_history, smooth_window=5)